# Pair 3 — XSimGCL: Amazon Product Recommendation

I evaluate XSimGCL using the same Amazon Musical Instruments data as LightGCL: 10,000 users and 17,233 products. XSimGCL uses graph propagation and noise-perturbed embeddings for contrastive learning, without graph edge augmentation or SVD.

This notebook follows the authors' main architecture and objectives, but adapts the data loading, negative sampling and evaluation for my dataset. It is not an exact reproduction of their experiments.

Reference: [XSimGCL — SELFRec authors' code](https://github.com/Coder-Yu/SELFRec/blob/main/model/graph/XSimGCL.py) · [Paper](https://arxiv.org/abs/2209.02544).

## 1. Setup


In [1]:
from pathlib import Path
import json, random, time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

BASE = Path.cwd()
if not (BASE / "datasets" / "processed" / "train.csv").exists():
    BASE = BASE / "Models" / "Pair_03"
DATA = BASE / "datasets" / "processed"
OUT = BASE / "results" / "XSimGCL"
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PILOT = False

DIM = 128
LAYERS = 2
CL_LAYER = 1     # Layer used for the contrastive view (1-based)
EPS = 0.1        # Noise strength
TEMP = 0.2       # Contrastive temperature
LAMBDA_CL = 0.05  # Contrastive-loss weight (starting point)
LAMBDA_L2 = 1e-4
LR = 0.001
BATCH_SIZE = 1024
EPOCHS = 3 if PILOT else 50
PATIENCE = 5
assert 1 <= CL_LAYER <= LAYERS
print("Device:", DEVICE, "| Data:", DATA)

Device: cuda | Data: c:\Users\royay\OneDrive\Desktop\COMP8410\Assignment 1\COMP8410_A1_49128299\Models\Pair_03\datasets\processed


## 2. Training graph

I construct the user-item graph from training interactions only. Validation and test interactions are held out. The processed user and item IDs must be consecutive integers starting from zero.

In [2]:
train = pd.read_csv(DATA / "train.csv")
valid = pd.read_csv(DATA / "valid.csv")
test = pd.read_csv(DATA / "test.csv")

for name, df in [("train", train), ("valid", valid), ("test", test)]:
    assert {"user_idx", "item_idx"}.issubset(df.columns), name
    assert not df[["user_idx", "item_idx"]].isna().any().any()
    df["user_idx"] = df["user_idx"].astype(int)
    df["item_idx"] = df["item_idx"].astype(int)

N_USERS = int(train.user_idx.max()) + 1
N_ITEMS = int(train.item_idx.max()) + 1
assert np.array_equal(np.sort(train.user_idx.unique()), np.arange(N_USERS))
assert np.array_equal(np.sort(train.item_idx.unique()), np.arange(N_ITEMS))
for df in (valid, test):
    assert df.user_idx.between(0, N_USERS - 1).all()
    assert df.item_idx.between(0, N_ITEMS - 1).all()

R = sp.coo_matrix(
    (np.ones(len(train), dtype=np.float32), (train.user_idx, train.item_idx)),
    shape=(N_USERS, N_ITEMS)
).tocsr()
R.data[:] = 1.0

# Symmetric normalisation: D^(-1/2) A D^(-1/2)
zero = sp.csr_matrix((N_USERS, N_USERS), dtype=np.float32)
zero_i = sp.csr_matrix((N_ITEMS, N_ITEMS), dtype=np.float32)
G = sp.bmat([[zero, R], [R.T, zero_i]], format="csr")
degree = np.asarray(G.sum(axis=1)).ravel()
d_inv = np.power(np.maximum(degree, 1), -0.5)
G = (sp.diags(d_inv) @ G @ sp.diags(d_inv)).tocoo()
indices = torch.from_numpy(np.vstack([G.row, G.col]).astype(np.int64))
values = torch.from_numpy(G.data.astype(np.float32))
ADJ = torch.sparse_coo_tensor(indices, values, G.shape).coalesce().to(DEVICE)

seen = [set(R.getrow(u).indices.tolist()) for u in range(N_USERS)]
print("Users:", N_USERS, "| Items:", N_ITEMS, "| Training interactions:", R.nnz)
print("Validation:", len(valid), "| Test:", len(test))

Users: 10000 | Items: 17233 | Training interactions: 59681
Validation: 7425 | Test: 7148


## 3. XSimGCL model

The model propagates embeddings through the interaction graph. During training, it adds normalised random noise at each layer and compares the final representation with an intermediate layer using InfoNCE. Recommendation loss uses BPR with negative items.

In [3]:
class XSimGCL(nn.Module):
    def __init__(self, n_users, n_items, dim):
        super().__init__()
        self.user = nn.Parameter(torch.empty(n_users, dim))
        self.item = nn.Parameter(torch.empty(n_items, dim))
        nn.init.xavier_uniform_(self.user)
        nn.init.xavier_uniform_(self.item)

    def forward(self, perturbed=False):
        emb = torch.cat([self.user, self.item], dim=0)
        layers = []
        cl_view = None
        for layer in range(1, LAYERS + 1):
            emb = torch.sparse.mm(ADJ, emb)
            if perturbed:
                noise = F.normalize(torch.rand_like(emb), dim=1)
                emb = emb + EPS * torch.sign(emb) * noise
            layers.append(emb)
            if layer == CL_LAYER:
                cl_view = emb
        final = torch.stack(layers, dim=0).mean(dim=0)
        final_u, final_i = torch.split(final, [N_USERS, N_ITEMS], dim=0)
        if perturbed:
            cl_u, cl_i = torch.split(cl_view, [N_USERS, N_ITEMS], dim=0)
            return final_u, final_i, cl_u, cl_i
        return final_u, final_i

    def loss(self, users, positives, negatives):
        eu, ei, cu, ci = self.forward(perturbed=True)
        pos_scores = (eu[users] * ei[positives]).sum(dim=1)
        neg_scores = (eu[users] * ei[negatives]).sum(dim=1)
        bpr = F.softplus(neg_scores - pos_scores).mean()

        # InfoNCE uses unique users and positive items within this batch.
        u = torch.unique(users)
        i = torch.unique(positives)
        def contrastive(a, b):
            a = F.normalize(a, dim=1)
            b = F.normalize(b, dim=1)
            return F.cross_entropy((a @ b.T) / TEMP,
                                   torch.arange(len(a), device=a.device))
        cl = contrastive(eu[u], cu[u]) + contrastive(ei[i], ci[i])
        reg = LAMBDA_L2 * (self.user[users].square().sum(dim=1).mean()
                           + self.item[positives].square().sum(dim=1).mean())
        return bpr + LAMBDA_CL * cl + reg, bpr, cl

model = XSimGCL(N_USERS, N_ITEMS, DIM).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print("Parameters:", sum(p.numel() for p in model.parameters()))

Parameters: 3485824


## 4. Full-catalog evaluation

I rank all 17,233 training-catalogue items and remove products each user has already interacted with in training. I use the same Recall@10/20, NDCG@10/20 and coverage definitions as my LightGCL notebook.

In [4]:
@torch.no_grad()
def evaluate(df, return_details=False):
    model.eval()
    eu, ei = model.forward(perturbed=False)
    targets = df.groupby("user_idx")["item_idx"].apply(set).to_dict()
    results, recommended = [], set()
    ids_all = list(targets)
    for start in range(0, len(ids_all), 256):
        ids = ids_all[start:start + 256]
        uid = torch.tensor(ids, dtype=torch.long, device=DEVICE)
        scores = eu[uid] @ ei.T
        for j, u in enumerate(ids):
            scores[j, list(seen[u])] = -float("inf")
        top = scores.topk(20, dim=1).indices.cpu().numpy()
        for u, ranked in zip(ids, top):
            recommended.update(map(int, ranked))
            truth = targets[u]
            rec = {"user_idx": int(u)}
            for k in (10, 20):
                hits = np.array([int(int(x) in truth) for x in ranked[:k]])
                rec[f"recall_at_{k}"] = float(hits.sum() / len(truth))
                dcg = float((hits / np.log2(np.arange(2, k + 2))).sum())
                ideal = float((1 / np.log2(np.arange(2, min(k, len(truth)) + 2))).sum())
                rec[f"ndcg_at_{k}"] = dcg / ideal if ideal else 0.0
            if return_details:
                rec["target_items"] = " ".join(map(str, sorted(truth)))
                rec["top20_items"] = " ".join(map(str, ranked))
            results.append(rec)
    detail = pd.DataFrame(results)
    cols = [f"{m}_at_{k}" for m in ("recall", "ndcg") for k in (10, 20)]
    summary = {col: float(detail[col].mean()) for col in cols}
    summary["coverage_at_20"] = len(recommended) / N_ITEMS
    summary["evaluated_users"] = len(detail)
    return summary, detail

def make_negatives(users, rng):
    negatives = rng.integers(0, N_ITEMS, size=len(users))
    for j, u in enumerate(users):
        while int(negatives[j]) in seen[int(u)]:
            negatives[j] = rng.integers(0, N_ITEMS)
    return negatives

## 5. Train and select the best model

I use validation NDCG@20 to select the checkpoint and stop training if it stops improving. The pilot uses three epochs and four batches per epoch.

In [5]:
rng = np.random.default_rng(SEED)
train_users = train.user_idx.to_numpy(dtype=np.int64)
train_items = train.item_idx.to_numpy(dtype=np.int64)
history = []
best_score, best_epoch, stale = -1.0, 0, 0
checkpoint = OUT / ("xsimgcl_pilot.pt" if PILOT else "xsimgcl_best.pt")
start_time = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    indices = rng.permutation(len(train_users))
    negatives = make_negatives(train_users, rng)
    losses = []
    for j, begin in enumerate(range(0, len(indices), BATCH_SIZE)):
        if PILOT and j >= 4:
            break
        idx = indices[begin:begin + BATCH_SIZE]
        users = torch.tensor(train_users[idx], device=DEVICE)
        positives = torch.tensor(train_items[idx], device=DEVICE)
        negs = torch.tensor(negatives[idx], device=DEVICE)
        model.train()
        optimizer.zero_grad()
        total, bpr, cl = model.loss(users, positives, negs)
        total.backward()
        optimizer.step()
        losses.append(float(total.detach().cpu()))

    validation, _ = evaluate(valid)
    score = validation["ndcg_at_20"]
    history.append({"epoch": epoch, "loss": float(np.mean(losses)),
                    "val_ndcg_at_20": score})
    print(f"Epoch {epoch:02d}: loss={np.mean(losses):.4f}, val NDCG@20={score:.4f}")
    if score > best_score + 1e-7:
        best_score, best_epoch, stale = score, epoch, 0
        torch.save(model.state_dict(), checkpoint)
    else:
        stale += 1
        if stale >= PATIENCE:
            print("Early stopping")
            break

train_seconds = time.perf_counter() - start_time
suffix = "_pilot" if PILOT else ""
pd.DataFrame(history).to_csv(OUT / f"history{suffix}.csv", index=False)
print(f"Best epoch: {best_epoch} | Validation NDCG@20: {best_score:.4f} | Training: {train_seconds:.1f}s")

Epoch 01: loss=0.9340, val NDCG@20=0.0121
Epoch 02: loss=0.8881, val NDCG@20=0.0185
Epoch 03: loss=0.8719, val NDCG@20=0.0242
Epoch 04: loss=0.8492, val NDCG@20=0.0282
Epoch 05: loss=0.8131, val NDCG@20=0.0319
Epoch 06: loss=0.7599, val NDCG@20=0.0348
Epoch 07: loss=0.6900, val NDCG@20=0.0363
Epoch 08: loss=0.6126, val NDCG@20=0.0376
Epoch 09: loss=0.5388, val NDCG@20=0.0379
Epoch 10: loss=0.4755, val NDCG@20=0.0381
Epoch 11: loss=0.4240, val NDCG@20=0.0383
Epoch 12: loss=0.3845, val NDCG@20=0.0386
Epoch 13: loss=0.3534, val NDCG@20=0.0385
Epoch 14: loss=0.3299, val NDCG@20=0.0385
Epoch 15: loss=0.3107, val NDCG@20=0.0385
Epoch 16: loss=0.2961, val NDCG@20=0.0382
Epoch 17: loss=0.2844, val NDCG@20=0.0379
Early stopping
Best epoch: 12 | Validation NDCG@20: 0.0386 | Training: 30.7s


## 6. Test results

I evaluate the checkpoint selected using validation data on the held-out test set. Outputs are saved in `results/XSimGCL/` for the final comparison.

In [6]:
model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
t0 = time.perf_counter()
test_metrics, details = evaluate(test, return_details=True)
test_seconds = time.perf_counter() - t0
metrics = {**test_metrics, "best_epoch": best_epoch,
           "best_val_ndcg_at_20": best_score, "train_seconds": train_seconds,
           "test_seconds": test_seconds, "pilot": PILOT,
           "dim": DIM, "layers": LAYERS, "cl_layer": CL_LAYER,
           "eps": EPS, "temperature": TEMP, "lambda_cl": LAMBDA_CL,
           "lambda_l2": LAMBDA_L2, "lr": LR, "batch_size": BATCH_SIZE,
           "seed": SEED, "device": str(DEVICE)}
print(pd.Series(metrics).to_string())
with open(OUT / f"metrics{suffix}.json", "w") as f:
    json.dump(metrics, f, indent=2)
details.to_csv(OUT / f"test_recommendations{suffix}.csv", index=False)

h = pd.DataFrame(history)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(h.epoch, h.val_ndcg_at_20, marker="o")
ax.set(xlabel="Epoch", ylabel="Validation NDCG@20", title="XSimGCL validation performance")
plt.tight_layout()
plt.savefig(OUT / f"validation_curve{suffix}.png", dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(h.epoch, h.loss, marker="o")
ax.set(xlabel="Epoch", ylabel="Training loss", title="XSimGCL training loss")
plt.tight_layout()
plt.savefig(OUT / f"training_loss{suffix}.png", dpi=160)
plt.show()

recall_at_10            0.046167
recall_at_20            0.066732
ndcg_at_10              0.024952
ndcg_at_20              0.030151
coverage_at_20          0.417107
evaluated_users             7148
best_epoch                    12
best_val_ndcg_at_20     0.038582
train_seconds          30.699301
test_seconds            0.961253
pilot                      False
dim                          128
layers                         2
cl_layer                       1
eps                          0.1
temperature                  0.2
lambda_cl                   0.05
lambda_l2                 0.0001
lr                         0.001
batch_size                  1024
seed                          42
device                      cuda


<Figure size 700x400 with 1 Axes>

<Figure size 700x400 with 1 Axes>

## Results and Discussion

I tested different contrastive-loss weights and embedding dimensions for XSimGCL using the Amazon Musical Instruments dataset. The best configuration used 128 embedding dimensions, two graph layers and a contrastive weight of 0.05. It achieved its highest validation NDCG@20 of 0.0386 at epoch 12, with early stopping at epoch 17.

On the test set, the model achieved Recall@10 of 0.0462, Recall@20 of 0.0667, NDCG@10 of 0.0250 and NDCG@20 of 0.0302. Item coverage@20 was 41.71%, and training took approximately 30.7 seconds using CUDA.

Reducing the contrastive-loss weight from 0.1 to 0.05 improved validation performance, while increasing the embedding dimension from 64 to 128 further improved recommendation accuracy and catalogue coverage. This suggests that the larger embeddings helped the model capture more useful user-product relationships.

However, overall recommendation accuracy remained relatively low, with Recall@20 indicating that only about 6.7% of held-out positive products appeared in the top 20 recommendations. The sparse interaction history, large product catalogue and limited hyperparameter testing may have affected performance. The results are also based on one random seed and a 10,000-user subset, so they may not generalise to the full Amazon dataset.